# Net2Vec Model Training on 10K sentences

In [1]:
import os
import time
from pathlib import Path
from typing import Iterator, List
import json
from datetime import datetime

from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from tqdm import tqdm

In [2]:
class TraceSentences:
    """Memory-efficient iterator for PCP traces"""
    
    def __init__(self, filepath: str, max_traces: int = None, show_progress: bool = True):
        self.filepath = filepath
        self.max_traces = max_traces
        self.show_progress = show_progress
        
        # Count total lines
        print(f"📊 Counting traces in {Path(filepath).name}...")
        with open(filepath, 'r', encoding='utf-8') as f:
            self.total_traces = sum(1 for _ in f)
        
        if max_traces:
            self.total_traces = min(self.total_traces, max_traces)
        
        print(f"  Total traces: {self.total_traces:,}")
        print(f"  File size: {os.path.getsize(filepath) / (1024**3):.2f} GB\n")
    
    def __iter__(self) -> Iterator[List[str]]:
        """Iterate over traces, yielding tokenized sentences"""
        with open(self.filepath, 'r', encoding='utf-8') as f:
            
            # Add progress bar for iteration
            iterator = tqdm(
                f, 
                total=self.total_traces,
                desc="Processing traces",
                unit=" traces",
                unit_scale=True,
                disable=not self.show_progress
            ) if self.show_progress else f
            
            for i, line in enumerate(iterator):
                if self.max_traces and i >= self.max_traces:
                    break
                
                tokens = line.strip().split()
                if tokens:
                    yield tokens    

In [3]:
class TrainingCallback(CallbackAny2Vec):
    """Enhanced callback with time estimation"""
    
    def __init__(self, total_epochs: int, checkpoint_dir: str = "model_checkpoints"):
        self.epoch = 0
        self.total_epochs = total_epochs
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        self.start_time = time.time()
        self.epoch_times = []
    
    def on_epoch_begin(self, model):
        """Called at start of each epoch"""
        self.epoch_start = time.time()
    
    def on_epoch_end(self, model):
        """Called at end of each epoch"""
        self.epoch += 1
        epoch_time = time.time() - self.epoch_start
        self.epoch_times.append(epoch_time)
        
        elapsed = time.time() - self.start_time
        avg_epoch_time = sum(self.epoch_times) / len(self.epoch_times)
        remaining_epochs = self.total_epochs - self.epoch
        eta = remaining_epochs * avg_epoch_time
        
        print(f"\n{'='*70}")
        print(f"✓ Epoch {self.epoch}/{self.total_epochs} complete")
        print(f"  Epoch time: {epoch_time:.1f}s")
        print(f"  Total elapsed: {elapsed:.1f}s ({elapsed/60:.1f} min)")
        print(f"  ETA: {eta:.1f}s ({eta/60:.1f} min)")
        print(f"{'='*70}")
        
        # Save checkpoint
        checkpoint_path = self.checkpoint_dir / f"word2vec_epoch{self.epoch}.model"
        model.save(str(checkpoint_path))
        print(f"💾 Checkpoint: {checkpoint_path.name}\n")

In [4]:
class Word2VecTrainer:
    """Production Word2Vec trainer with progress tracking"""
    
    def __init__(
        self,
        traces_file: str,
        output_dir: str = "trained_models",
        vector_size: int = 100,
        window: int = 5,
        min_count: int = 1,
        workers: int = 4,
        epochs: int = 5,
        sg: int = 1,
        negative: int = 5,
        max_traces: int = None
    ):
        self.traces_file = traces_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        self.params = {
            'vector_size': vector_size,
            'window': window,
            'min_count': min_count,
            'workers': workers,
            'epochs': epochs,
            'sg': sg,
            'negative': negative,
            'max_traces': max_traces
        }
        
        self.model = None
        self.training_stats = {}
    
    def train(self):
        """Train Word2Vec model with progress tracking"""
        print("\n" + "="*80)
        print("WORD2VEC TRAINING - FULL PRODUCTION MODEL")
        print("="*80)
        print(f"\n Input: {self.traces_file}")
        print(f"Output: {self.output_dir}")
        print(f"\n Hyperparameters:")
        for k, v in self.params.items():
            if k != 'max_traces' or v is not None:
                print(f"  {k}: {v}")
        print("="*80 + "\n")
        
        start_time = time.time()
        
        # Create sentence iterator
        print("🔄 Creating trace iterator...")
        sentences = TraceSentences(
            self.traces_file,
            max_traces=self.params['max_traces'],
            show_progress=False  # Progress shown during training
        )
        
        # Create callback
        callback = TrainingCallback(
            total_epochs=self.params['epochs']
        )
        
        # Build vocabulary
        print("\n📚 Building vocabulary...")
        print("   (This scans all traces once - may take 3-5 minutes)\n")
        vocab_start = time.time()
        
        self.model = Word2Vec(
            vector_size=self.params['vector_size'],
            window=self.params['window'],
            min_count=self.params['min_count'],
            workers=self.params['workers'],
            sg=self.params['sg'],
            negative=self.params['negative'],
            epochs=self.params['epochs'],
            callbacks=[callback]
        )
        
        # Build vocab with progress
        vocab_sentences = TraceSentences(
            self.traces_file,
            max_traces=self.params['max_traces'],
            show_progress=True
        )
        self.model.build_vocab(vocab_sentences)
        
        vocab_time = time.time() - vocab_start
        
        print(f"\n✓ Vocabulary built!")
        print(f"  Vocabulary size: {len(self.model.wv):,} unique PCP words")
        print(f"  Time: {vocab_time:.1f}s ({vocab_time/60:.1f} min)\n")
        
        # Train model
        print("🚀 Training Word2Vec model...")
        print(f"   {self.params['epochs']} epochs × {sentences.total_traces:,} traces")
        print(f"   Estimated time: {self.params['epochs'] * 10}-{self.params['epochs'] * 15} minutes\n")
        
        train_start = time.time()
        
        # Train with progress
        train_sentences = TraceSentences(
            self.traces_file,
            max_traces=self.params['max_traces'],
            show_progress=True
        )
        
        self.model.train(
            train_sentences,
            total_examples=sentences.total_traces,
            epochs=self.params['epochs']
        )
        
        train_time = time.time() - train_start
        
        # Save final model
        print("\n💾 Saving models...")
        final_model_path = self.output_dir / "word2vec_final.model"
        self.model.save(str(final_model_path))
        
        vectors_path = self.output_dir / "word2vec_vectors.kv"
        self.model.wv.save(str(vectors_path))
        
        # Save embedding matrix as numpy
        import numpy as np
        embedding_matrix = self.model.wv.vectors
        matrix_path = self.output_dir / "embedding_matrix.npy"
        np.save(str(matrix_path), embedding_matrix)
        
        # Save vocabulary
        vocab_path = self.output_dir / "vocabulary.txt"
        with open(vocab_path, 'w', encoding='utf-8') as f:
            for word in self.model.wv.index_to_key:
                f.write(f"{word}\n")
        
        # Statistics
        total_time = time.time() - start_time
        self.training_stats = {
            'vocab_size': len(self.model.wv),
            'total_traces': sentences.total_traces,
            'vector_size': self.params['vector_size'],
            'vocab_build_time': vocab_time,
            'training_time': train_time,
            'total_time': total_time,
            'traces_per_second': sentences.total_traces * self.params['epochs'] / train_time,
            'hyperparameters': self.params,
            'timestamp': datetime.now().isoformat()
        }
        
        stats_path = self.output_dir / "training_stats.json"
        with open(stats_path, 'w') as f:
            json.dump(self.training_stats, f, indent=2)
        
        self._print_summary(final_model_path, vectors_path, matrix_path)
    
    def _print_summary(self, model_path, vectors_path, matrix_path):
        """Print training summary"""
        print("\n" + "="*80)
        print("🎉 TRAINING COMPLETE!")
        print("="*80)
        print(f"\n📊 Final Statistics:")
        print(f"  Vocabulary: {self.training_stats['vocab_size']:,} unique PCP words")
        print(f"  Total traces: {self.training_stats['total_traces']:,}")
        print(f"  Vector dimension: {self.training_stats['vector_size']}")
        print(f"  Vocab build: {self.training_stats['vocab_build_time']:.1f}s ({self.training_stats['vocab_build_time']/60:.1f} min)")
        print(f"  Training: {self.training_stats['training_time']:.1f}s ({self.training_stats['training_time']/60:.1f} min)")
        print(f"  Total: {self.training_stats['total_time']:.1f}s ({self.training_stats['total_time']/60:.1f} min)")
        print(f"  Processing rate: {self.training_stats['traces_per_second']:,.0f} traces/sec")
        
        print(f"\n💾 Saved Files:")
        print(f"  ✓ Full model: {model_path.name}")
        print(f"  ✓ Vectors only: {vectors_path.name}")
        print(f"  ✓ NumPy matrix: {matrix_path.name}")
        print(f"  ✓ Vocabulary: vocabulary.txt")
        print(f"  ✓ Statistics: training_stats.json")
        print(f"  ✓ Checkpoints: model_checkpoints/")
        print("="*80 + "\n")
    
    def test_embeddings(self, num_examples: int = 10):
        """Test trained embeddings"""
        if not self.model:
            print("Error: Model not trained!")
            return
        
        print("\n" + "="*80)
        print("TESTING EMBEDDINGS")
        print("="*80)
        
        # Get sample words
        sample_words = list(self.model.wv.index_to_key[:100])
        
        # Find interesting patterns
        and_gates = [w for w in sample_words if 'nnd' in w.lower() or 'and' in w.lower()]
        or_gates = [w for w in sample_words if 'nor' in w.lower() or 'or' in w.lower()]
        
        test_words = (and_gates[:3] + or_gates[:3])[:num_examples]
        
        for word in test_words:
            print(f"\n🔍 Most similar to: {word}")
            similar = self.model.wv.most_similar(word, topn=5)
            for sim_word, score in similar:
                print(f"  {sim_word:40s} {score:.4f}")
        
        print("\n" + "="*80)

In [ ]:
# """Main training function"""
    
#     # Check file
#     TRACES_FILE = "vocabulary_traces.txt"
#     if not os.path.exists(TRACES_FILE):
#         print(f"❌ Error: {TRACES_FILE} not found!")
#         return
    
#     # Configuration
#     trainer = Word2VecTrainer(
#         traces_file=TRACES_FILE,
#         output_dir="trained_models",
#         vector_size=100,      # Paper standard
#         window=5,             # Context window
#         min_count=1,          # Keep all PCP words
#         workers=4,            # CPU threads (adjust for your CPU)
#         epochs=5,             # Paper uses 5
#         sg=1,                 # Skip-gram
#         negative=5,           # Negative sampling
#         max_traces=None       # None = all 59.6M traces
#     )
    
#     # Train
#     print("\n⚠️  FULL TRAINING - This will take 45-65 minutes!")
#     print("    You can interrupt (Ctrl+C) and resume from checkpoints.\n")
    
#     input("Press Enter to start training...")
    
#     trainer.train()
#     trainer.test_embeddings()
    
#     print("\n✅ Training complete!")
#     print("   Next: Translate traces to vectors for classification\n")



trainer = Word2VecTrainer(
    traces_file=r"E:\University of Kerman\Term 9\VHDL\Paper Implementation\DL_trojan_detection\vocabulary_traces_small.txt",
    output_dir="test_models",
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=4,
    sg=1,
    negative=5,
    max_traces=None  # all the traces!
)
    


## Train Word2Vec

In [8]:
trainer.train()


WORD2VEC TRAINING - FULL PRODUCTION MODEL

 Input: E:\University of Kerman\Term 9\VHDL\Paper Implementation\DL_trojan_detection\vocabulary_traces_small.txt
Output: test_models

 Hyperparameters:
  vector_size: 100
  window: 5
  min_count: 1
  workers: 4
  epochs: 4
  sg: 1
  negative: 5

🔄 Creating trace iterator...
📊 Counting traces in vocabulary_traces_small.txt...
  Total traces: 1,342,144
  File size: 0.09 GB


📚 Building vocabulary...
   (This scans all traces once - may take 3-5 minutes)

📊 Counting traces in vocabulary_traces_small.txt...
  Total traces: 1,342,144
  File size: 0.09 GB



Processing traces: 100%|██████████| 1.34M/1.34M [00:02<00:00, 471k traces/s]



✓ Vocabulary built!
  Vocabulary size: 383 unique PCP words
  Time: 3.3s (0.1 min)

🚀 Training Word2Vec model...
   4 epochs × 1,342,144 traces
   Estimated time: 40-60 minutes

📊 Counting traces in vocabulary_traces_small.txt...
  Total traces: 1,342,144
  File size: 0.09 GB



Processing traces: 100%|██████████| 1.34M/1.34M [00:05<00:00, 252k traces/s]
Processing traces: 100%|██████████| 1.34M/1.34M [00:05<00:00, 252k traces/s]
Processing traces: 100%|██████████| 1.34M/1.34M [00:05<00:00, 235k traces/s]
Processing traces: 100%|██████████| 1.34M/1.34M [00:05<00:00, 230k traces/s]



💾 Saving models...

🎉 TRAINING COMPLETE!

📊 Final Statistics:
  Vocabulary: 383 unique PCP words
  Total traces: 1,342,144
  Vector dimension: 100
  Vocab build: 3.3s (0.1 min)
  Training: 22.8s (0.4 min)
  Total: 26.6s (0.4 min)
  Processing rate: 235,325 traces/sec

💾 Saved Files:
  ✓ Full model: word2vec_final.model
  ✓ Vectors only: word2vec_vectors.kv
  ✓ NumPy matrix: embedding_matrix.npy
  ✓ Vocabulary: vocabulary.txt
  ✓ Statistics: training_stats.json
  ✓ Checkpoints: model_checkpoints/



## Test Word2Vec

In [9]:
trainer.test_embeddings()


TESTING EMBEDDINGS

🔍 Most similar to: Q_nnd2s1_Q
  Q_or4s1_DIN5                             0.4345
  DIN3_nnd2s1_DIN3                         0.4184
  Q_or3s1_DIN4                             0.3909
  Q_or4s1_Q                                0.3872
  Q_or4s1_DIN4                             0.3747

🔍 Most similar to: DIN1_nnd2s1_DIN2
  DIN1_nnd2s1_DIN1                         0.4408
  DIN1_nnd2s1_DIN                          0.4294
  DIN1_xor2s1_DIN                          0.3902
  Q_or3s1_DIN3                             0.3776
  DIN1_or4s1_DIN                           0.3567

🔍 Most similar to: DIN2_nnd2s1_DIN1
  DIN2_nnd2s1_DIN2                         0.4682
  DIN1_i1s1_DIN1                           0.4163
  DIN2_nnd2s1_DIN                          0.3805
  DIN2_hi1s1_DIN1                          0.3631
  Q_or4s1_DIN5                             0.3577

🔍 Most similar to: Q_nor2s1_Q
  Q_xor2s1_Q                               0.4494
  Q_and4s1_Q                               0